read from translated top sun list that look like this: wildtype_aa1	pos1	mutate_aa1	wildtype_aa2	pos2	mutate_aa2	DDE	untranslated_wildtype_aa1	untranslated_mutate_aa1	untranslated_wildtype_aa2	untranslated_mutate_aa2
B	30	D	B	88	C	4.6108785	D	N	N	D
C	32	D	A	47	B	3.4506056	V	I	I	V
B	48	A	C	54	A	3.1376057	G	V	I	A
B	30	D	C	45	D	3.081643	D	N	K	Q,  get the list of double mutation string in format of "D30N-N88D"

In [34]:
import pandas as pd
from io import StringIO

# Read the data into a DataFrame from a TSV file
df = pd.read_csv("data/translated_top_syn_list.tsv", sep="\t")

# Generate the double mutation strings
double_mutations = df.apply(
    lambda row: f"{row['untranslated_wildtype_aa1']}{row['pos1']}{row['untranslated_mutate_aa1']}_"
                f"{row['untranslated_wildtype_aa2']}{row['pos2']}{row['untranslated_mutate_aa2']}", axis=1)

# Convert to a list
double_mutation_list = double_mutations.tolist()
double_mutation_tuples = list(zip(double_mutation_list, df['DDE']))
print(double_mutation_tuples)

[('D30N_N88D', 4.6108785), ('V32I_I47V', 3.4506056), ('G48V_I54A', 3.1376057), ('D30N_K45Q', 3.081643), ('I54A_V82A', 2.9455953), ('I54V_V82A', 2.8622637), ('M46I_L76V', 2.665547), ('I54V_V82T', 2.476119), ('I54A_V82T', 2.3435872), ('G48V_V82A', 2.3191676), ('I54A_A71I', 2.26015), ('M46I_N88T', 2.2406816), ('L90M_C95F', 2.0020018), ('V32I_M46I', 1.9782121), ('G48V_V82T', 1.9567208), ('I54V_T91S', 1.9268732), ('M46I_F53Y', 1.8669852), ('M46L_K55R', 1.8074695), ('M46L_V82A', 1.8062705), ('M46I_K55R', 1.7659798)]


read the mutation from double_mutation tuple list and output like this in a tsv form:
                    
Mutation Pairs	ΔΔE	Energy Type	ΔE(M1,M2)	ΔE(M1)	ΔE(M2)
D30N-N88D		Wildtype			
D30N-N88D		Rescue example			
D30N-N88D		Compensate example			
D30N-N88D		Antagonistic example			
D30N-N88D		flip example			                    
V32I-I47V		

for each pair, Delta delta E is in the tuple. and each has a row of energy type fixed to the example above. To access dm12, dm1, dm2,read into corrosponding folder: for wildtype its data/wildtype_out/{mutation name}.tsv, for rescue its data/rescue_out/{mutation name}.tsv and so on, access the first rows value and put in a new summary.tsv file. 

In [ ]:
import os

# Define the energy types and their corresponding folders
energy_types = [
    ("Wildtype", "consensus_out"),
    ("Rescue example", "rescue_out"),
    ("Compensate example", "compensate_out"),
    ("Antagonistic example", "antag_out"),
    ("flip example", "flip_out"),
]

# Prepare the output data
output_data = []

for mutation, dde in double_mutation_tuples:
    for energy_type, folder in energy_types:
        # Construct the file path
        file_path = f"data/{folder}/{mutation}.tsv"
        
        # Read the first row value if the file exists
        if os.path.exists(file_path):
            with open(file_path, 'r') as file:
                first_row = file.readline().strip()
                second_row = file.readline().strip()
            # print(f"Read from {file_path}: {first_row}")
        else:
            # print(f"File not found: {file_path}")
            first_row = ""
        
        # Append the row to the output data
        first_row_values = first_row.split("\t")
        second_row_values = second_row.split("\t") if second_row else []
        output_data.append([mutation, dde, energy_type, second_row_values[5],second_row_values[3], second_row_values[4]])
    # print(output_data)

# Create a DataFrame for the output
output_df = pd.DataFrame(output_data, columns=["Mutation Pairs", "ΔΔE", "Energy Type", "ΔE(M1,M2)", "ΔE(M1)", "ΔE(M2)"])

# Save the DataFrame to a TSV file
output_df.to_csv("data/summary.tsv", sep="\t", index=False)

[['D30N_N88D', 4.6108785, 'Wildtype', '-4.8736153', '-3.3826494', '-6.101837'], ['D30N_N88D', 4.6108785, 'Rescue example', '2.182047', '-0.94966507', '-1.4791632'], ['D30N_N88D', 4.6108785, 'Compensate example', '-2.1058712', '-1.2527208', '-5.464025'], ['D30N_N88D', 4.6108785, 'Antagonistic example', '-5.6063213', '-4.6131735', '-5.604015'], ['D30N_N88D', 4.6108785, 'flip example', '0.60025597', '-2.047474', '-1.9631433']]
[['D30N_N88D', 4.6108785, 'Wildtype', '-4.8736153', '-3.3826494', '-6.101837'], ['D30N_N88D', 4.6108785, 'Rescue example', '2.182047', '-0.94966507', '-1.4791632'], ['D30N_N88D', 4.6108785, 'Compensate example', '-2.1058712', '-1.2527208', '-5.464025'], ['D30N_N88D', 4.6108785, 'Antagonistic example', '-5.6063213', '-4.6131735', '-5.604015'], ['D30N_N88D', 4.6108785, 'flip example', '0.60025597', '-2.047474', '-1.9631433'], ['V32I_I47V', 3.4506056, 'Wildtype', '-7.4023676', '-4.8156433', '-6.0373354'], ['V32I_I47V', 3.4506056, 'Rescue example', '6.372223', '3.777339